In [117]:
thetas = [[2,3], [1,2]]

for j, params in enumerate(thetas):

    print(j, params)

0 [2, 3]
1 [1, 2]


In [ ]:
from lotteries import lotteries_full
import functions as f
from main import get_observed_ce
import numpy as np
import importlib
import mixture
from scipy.stats import norm

importlib.reload(mixture)

from mixture import compute_log_likelihoods

In [119]:
lottery = lotteries_full

lotteries = f.transform(lotteries_full)

thetas=[[1, 2, 3, 4, 5, 6], [2, 3, 5, 7, 7]]

method = "tk"

c = 2

In [120]:
lotteries["lottery_14"]

{'name': 'Papaya',
 'spread': 83,
 'outcomes': {0: {0.06: [0, -9, -8, 47]},
  1: {0.14: [0, -9, -8, -22]},
  2: {0.08: [0, 9, -17, -31]},
  3: {0.08: [0, 9, -17, 40]},
  4: {0.384: [0, 9, -5, -22]},
  5: {0.256: [0, 9, -5, 40]}}}

In [121]:
y = get_observed_ce(export_excel=False)

subjects = sorted(y["participant_label"].unique())

n = len(subjects)

ksi = np.ones(n)*(1/2)

ksi_map = {subj: ksi[i] for i, subj in enumerate(subjects)}

spreads = {lid: lotteries[lid]["spread"] for lid in lotteries}

print(spreads)


{'lottery_1': 90, 'lottery_2': 106, 'lottery_3': 96, 'lottery_4': 1180, 'lottery_5': 1110, 'lottery_6': 1060, 'lottery_7': 1835, 'lottery_8': 2070, 'lottery_9': 1381, 'lottery_10': 87, 'lottery_11': 76, 'lottery_12': 85, 'lottery_13': 87, 'lottery_14': 83}


In [122]:
y = y.copy()

In [123]:
y["spread"] = y["lottery_id"].map(spreads)

In [124]:
y["sigma"] = y["participant_label"].map(ksi_map) * y["spread"]

In [125]:
grouped_y = y.groupby("participant_label", observed=True)

log_L = np.zeros((n, c))

In [126]:
for j, params in enumerate(thetas):

        if method == "tk":

            r, alpha, lamb, gamma = params[:4]

            beta, palpha = 1, 1

        elif method == "prelec":

            r, alpha, lamb, beta, palpha = params[:5]

            gamma = 0.61

        else:

            raise ValueError(f"Unknown method: {method!r}")


        R = 0

        ce_theoretical = f.ce_dict(r, gamma, alpha, lamb, R = R, lotteries=lotteries, method=method, beta=beta, palpha=palpha)

        for i, subj_label in enumerate(subjects):

            y_subj   = grouped_y.get_group(subj_label)

            loc_vals = y_subj["lottery_id"].map(ce_theoretical)

            log_L[i, j] = np.sum(norm.logpdf(y_subj["ce_observed"], loc=loc_vals, scale=y_subj["sigma"]))

            

In [127]:
log_L

array([[ -92.22984677,  -92.26819445],
       [ -97.58780018,  -97.81454833],
       [ -97.25330688,  -97.20678731],
       [-100.35385728, -100.26086466],
       [ -92.38884813,  -92.39567296],
       [ -98.37762343,  -98.43167841],
       [ -97.97728195,  -98.04845996],
       [ -95.63191543,  -95.41722665],
       [ -92.5306128 ,  -92.58721342],
       [ -97.78638015,  -97.77093125],
       [ -92.70743889,  -92.6408054 ],
       [ -98.01298145,  -97.98422443],
       [ -93.27337868,  -93.35768324],
       [-100.22947249, -100.19269314],
       [ -91.98271082,  -91.99166986],
       [ -93.42505533,  -93.32629886],
       [ -92.08821115,  -92.13601919],
       [ -97.69221241,  -97.63730301],
       [ -93.29979909,  -93.39629219],
       [ -97.83373817,  -97.7427821 ],
       [ -97.84184042,  -97.88308905],
       [ -93.295383  ,  -93.31630692],
       [ -97.15561614,  -97.16089013],
       [ -98.80326182,  -98.70132933],
       [-104.42193005, -104.61413555]])

In [128]:
compute_log_likelihoods(thetas=[[1, 2, 3, 4, 5, 6], [2, 3, 5, 7, 8, 5]], ksi=ksi, method="tk", c=2)

array([[ -92.22984677,  -92.26819445],
       [ -97.58780018,  -97.81454833],
       [ -97.25330688,  -97.20678731],
       [-100.35385728, -100.26086466],
       [ -92.38884813,  -92.39567296],
       [ -98.37762343,  -98.43167841],
       [ -97.97728195,  -98.04845996],
       [ -95.63191543,  -95.41722665],
       [ -92.5306128 ,  -92.58721342],
       [ -97.78638015,  -97.77093125],
       [ -92.70743889,  -92.6408054 ],
       [ -98.01298145,  -97.98422443],
       [ -93.27337868,  -93.35768324],
       [-100.22947249, -100.19269314],
       [ -91.98271082,  -91.99166986],
       [ -93.42505533,  -93.32629886],
       [ -92.08821115,  -92.13601919],
       [ -97.69221241,  -97.63730301],
       [ -93.29979909,  -93.39629219],
       [ -97.83373817,  -97.7427821 ],
       [ -97.84184042,  -97.88308905],
       [ -93.295383  ,  -93.31630692],
       [ -97.15561614,  -97.16089013],
       [ -98.80326182,  -98.70132933],
       [-104.42193005, -104.61413555]])